In [1]:
import pandas as pd
import plotly.graph_objects as go
from fredapi import Fred
from dotenv import load_dotenv
import os

load_dotenv()
fred = Fred(api_key=os.getenv('FRED_API_KEY'))

print("FRED API connected successfully!")

FRED API connected successfully!


In [4]:
# CPIAUCSL = Consumer Price Index for All Urban Consumers
# This is the most commonly used inflation metric in the US
cpi = fred.get_series('CPIAUCSL', observation_start='1965-01-01', observation_end='1985-12-31')

# Convert to year-over-year percentage change (annual inflation rate)
us_inflation = cpi.pct_change(periods=12) * 100

# Clean up into a dataframe
us_inflation = us_inflation.dropna().to_frame(name='inflation')
us_inflation.index.name = 'date'

print(f"Date range: {us_inflation.index[0].strftime('%Y-%m')} to {us_inflation.index[-1].strftime('%Y-%m')}")
print(f"Records: {len(us_inflation)}")
print(f"\nPeak inflation: {us_inflation['inflation'].max():.1f}% ({us_inflation['inflation'].idxmax().strftime('%Y-%m')})")
print(f"\nFirst 5 rows:")
us_inflation.head()

Date range: 1966-01 to 1985-12
Records: 240

Peak inflation: 14.6% (1980-03)

First 5 rows:


,inflation
date,
1966-01-01,1.918159
1966-02-01,2.557545
1966-03-01,2.778665
1966-04-01,2.868069
1966-05-01,2.763659


In [5]:
# UNRATE = US Civilian Unemployment Rate (monthly, seasonally adjusted)
us_unemployment = fred.get_series('UNRATE', observation_start='1965-01-01', observation_end='1985-12-31')

# Convert to dataframe
us_unemployment = us_unemployment.to_frame(name='unemployment')
us_unemployment.index.name = 'date'

print(f"Date range: {us_unemployment.index[0].strftime('%Y-%m')} to {us_unemployment.index[-1].strftime('%Y-%m')}")
print(f"Records: {len(us_unemployment)}")
print(f"\nPeak unemployment: {us_unemployment['unemployment'].max():.1f}% ({us_unemployment['unemployment'].idxmax().strftime('%Y-%m')})")
print(f"Lowest unemployment: {us_unemployment['unemployment'].min():.1f}% ({us_unemployment['unemployment'].idxmin().strftime('%Y-%m')})")

us_unemployment.head()

Date range: 1965-01 to 1985-12
Records: 252

Peak unemployment: 10.8% (1982-11)
Lowest unemployment: 3.4% (1968-09)


,unemployment
date,
1965-01-01,4.9
1965-02-01,5.1
1965-03-01,4.7
1965-04-01,4.8
1965-05-01,4.6


In [6]:
# Misery Index = Inflation + Unemployment
# Invented by economist Arthur Okun in the 1960s
# Simple but powerful: captures the "pain" an average citizen feels

us_misery = us_inflation.join(us_unemployment, how='inner')
us_misery['misery_index'] = us_misery['inflation'] + us_misery['unemployment']

print(f"Date range: {us_misery.index[0].strftime('%Y-%m')} to {us_misery.index[-1].strftime('%Y-%m')}")
print(f"Records: {len(us_misery)}")
print(f"\nPeak Misery Index: {us_misery['misery_index'].max():.1f} ({us_misery['misery_index'].idxmax().strftime('%Y-%m')})")
print(f"Lowest Misery Index: {us_misery['misery_index'].min():.1f} ({us_misery['misery_index'].idxmin().strftime('%Y-%m')})")

us_misery.head()

Date range: 1966-01 to 1985-12
Records: 240

Peak Misery Index: 21.9 (1980-05)
Lowest Misery Index: 5.9 (1966-01)


,inflation,unemployment,misery_index
date,,,
1966-01-01,1.918159,4.0,5.918159
1966-02-01,2.557545,3.8,6.357545
1966-03-01,2.778665,3.8,6.578665
1966-04-01,2.868069,3.8,6.668069
1966-05-01,2.763659,3.9,6.663659


In [8]:
# A191RL1Q225SBEA = Real GDP growth rate (quarterly, annualized)
# This is the standard measure - already calculated as % change by BEA
us_gdp = fred.get_series('A191RL1Q225SBEA', observation_start='1965-01-01', observation_end='1985-12-31')

us_gdp = us_gdp.to_frame(name='gdp_growth')
us_gdp.index.name = 'date'

print(f"Date range: {us_gdp.index[0].strftime('%Y-%m')} to {us_gdp.index[-1].strftime('%Y-%m')}")
print(f"Records: {len(us_gdp)}")
worst_date = us_gdp['gdp_growth'].idxmin()
print(f"\nWorst quarter: {us_gdp['gdp_growth'].min():.1f}% ({worst_date.year}-Q{worst_date.quarter})")
print(f"Best quarter: {us_gdp['gdp_growth'].max():.1f}% ({us_gdp['gdp_growth'].idxmax().strftime('%Y-%m')})")

us_gdp.head()

Date range: 1965-01 to 1985-10
Records: 84

Worst quarter: -8.0% (1980-Q2)
Best quarter: 16.4% (1978-04)


,gdp_growth
date,
1965-01-01,10.0
1965-04-01,5.2
1965-07-01,9.2
1965-10-01,9.5
1966-01-01,10.1
